# 04.4 · q-transfer — cross-task multi-predicate transplant

**Question.** Does the goal heads' *query* at the pivot carry "what to do next", and in what form? Transplant the SRC
pivot query into DST and see which task DST writes: one predicted by **set progress** (the query encodes *which* tasks
are done) or by **count** (it encodes *how many* are done).

**Notation.** Capital letters are tasks (predicates, knob `TASKS`); `'` = already written in the prefill; `→` = the
next task. Code variables are lettered **sequentially** (`a, b, c, …` in the order lines are written; the stub takes
the next letter) — unrelated to the task letters and never the task's list position.

**Configurations** (`USE_KV`):

| | prompts | DST writes (clean) | q-transfer predicts |
|---|---|---|---|
| `USE_KV=False` | `p_src A'CB' → C` · `p_dst A'BD → B` | B | set progress **D**, count **D** |
| `USE_KV=True` (04.3) | `p_src A'CB' → C` · `p_dst A'XY → X` · `p_kv A'BD → B` | X | kv-only **B** · kv+q: set **D**, count **Y** |

*Set progress* = first task of the list DST's goal heads read (p_kv's with kv, else DST's own) written in neither DST
nor SRC. *Count* = DST's slot `|SRC_DONE|` on DST's own listing. SRC's own target C is in neither p_dst nor p_kv.

**Pivot** (`PIVOT`): `"indent"` — the prefill ends at the trailing indent; `"var_eq"` — it ends at the next variable's
` =` (e.g. `    b =`), giving three anchors (indent, var, eq) on each side.

**Interventions** (goal heads; `N_GOAL_HEADS` = 122 or 250):
- **q-transfer** — SRC's pre-RoPE query at each of `PREFILL_ANCHORS` → the same anchor in DST (prefill), then one query
  (`DECODE_Q`) re-clamped for the first `N_STEER − 1` decode steps.
- **kv-transfer** (`USE_KV`) — p_kv's task-list key+value on `KV_HEADS` → DST's task-list span (GQA-corrected
  head-wise routing, `pad` alignment: newline tokens spliced before the shorter span, so both spans have the same
  length and every token moves); `KV_ALPHA` boosts the patched heads' OV write at the current token on every patched
  forward. The clean destination always runs on the unpadded DST.
- **closed loop** — instead of re-clamping one fixed query, regenerate the goal heads' query at every step from the
  goal encoding (see the closed-loop section).

**Sections:** prompts → runs (generation + GoalScope read-out) → causality read-out → gaze → closed loop
(+ its read-outs and per-step gaze).

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

CUDA = "7"                                   # free GPU(s); Coder-32B fits on one 80 GB GPU
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")                     # repo root, so `src` is importable
import json, torch, numpy as np
import src.diff.functional as FUNC
from src.diff.functional import *            # helpers into scope
import src.diff.data as DATA
from src.attention import get_attention_matrices
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.markup import escape
from src.utils.experiment_utils import set_seed

FUNC.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads the model + the 122 goal heads (lam20 · thr 0.9 · 25-56)
mt = FUNC.mt


def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift


def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)), title=f"[bold]{title}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

2026-10-06 15:00:18 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Knobs

In [36]:
# ── knobs ────────────────────────────────────────────────────────────────────
N_GOAL_HEADS = 250        # 122: lam20 · thr 0.9 · layers 25-56 (paper set) | 250: lam20 top-250 (best kv transfer)
DATATYPE     = "list[str]"
NAMING       = "letters"  # result variable names in the code: "letters" (a, b, c) | "natural" (descriptive)

TASKS = {                 # task letter -> predicate id in DATA.QTASKS[DATATYPE]
    "B": "count_long",
    "A": "shortest_word_length",
    "C": "distinct_count",
    "D": "count_starts_vowel",
    "X": "total_length",       # x: DST's own frontier (word-length family: distinct from b / c / d, as in 04.3)
    "Y": "has_palindrome",     # y: DST's 2nd pending task
}
# Design (04.3 notation, ' = written):
#   USE_KV=False:  p_src a'cb' -> c   p_dst a'bd -> b                      q-transfer: set progress -> d, count -> d
#   USE_KV=True :  p_src a'cb' -> c   p_dst a'xy -> x   p_kv a'bd -> b     kv-only -> b · kv+q: set -> d, count -> y
USE_KV   = False          # True: also transplant KV's task-list k+v onto DST's task list (04.3 setup)
SRC_ORDER, SRC_DONE = "ACB", "AB"   # SRC: listed tasks (in order), written tasks -> target = first listed not written
DST_ORDER, DST_DONE = ("AXY", "A") if USE_KV else ("ABD", "A")   # DST: listed tasks, written tasks -> frontier = first not written

PIVOT    = "var_eq"       # "indent": prefill ends at the trailing indent · "var_eq": ends at the next var's " =" (`c =`)
PREFILL_ANCHORS = ("indent", "var", "eq")   # prefill: SRC query at each anchor -> the SAME anchor in DST ("var"/"eq" need PIVOT="var_eq")
DECODE_Q = "eq"          # query re-clamped at decode steps: "eq" (SRC " =" query) | "mean" (avg of SRC indent/var/eq queries)

# PIVOT = "indent"
# PREFILL_ANCHORS = ("indent",)
# DECODE_Q = "indent"

N_STEER  = 32             # forward passes that carry the SRC query: 1 = DST pivot only (prefill); k = pivot + first k-1 decode steps
MAX_NEW  = 96             # generation budget per run
KV_ORDER = "ABD"          # KV's task list; its written tasks = DST_DONE, so only the task list differs from DST
KV_HEADS = None           # heads whose k+v are transplanted (None = GOAL_HEADS)
KV_ALPHA = 2.0            # OV-write boost of KV_HEADS ∪ PATCH_HEADS at the current token on patched runs (1.0 = off)
KV_ALIGN = "pad"          # kv span alignment: "pad" | "right"
RO_KW    = dict(strategy="additive_norm", alpha=0.1, layers=(35, 46))   # GoalScope read-out config (paper default)

# ── derived ──────────────────────────────────────────────────────────────────
if N_GOAL_HEADS == 122:
    GOAL_HEADS = list(FUNC.GOAL_HEADS)
elif N_GOAL_HEADS == 250:
    GOAL_HEADS = [tuple(h) for h in json.load(open(
        "../../head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json"))]
else:
    raise ValueError("N_GOAL_HEADS must be 122 or 250")

READOUT_HEADS = GOAL_HEADS

#! overwrite goal heads
# GOAL_HEADS = [(l, h) for l in range(35, 55) for h in range(mt.n_heads)]

PATCH_HEADS = GAZE_HEADS = GOAL_HEADS 
KV_HEADS = KV_HEADS or GOAL_HEADS

bank = DATA.QTASKS[DATATYPE]
by = {p.id: p for p in bank.predicates}
assert all(TASKS[L] in by for L in SRC_ORDER + DST_ORDER), "unknown predicate id in TASKS"
assert set(SRC_DONE) <= set(SRC_ORDER) and set(DST_DONE) <= set(DST_ORDER)
TARGET   = next(L for L in SRC_ORDER if L not in SRC_DONE)    # SRC's next task
FRONTIER = next(L for L in DST_ORDER if L not in DST_DONE)    # DST's own next task
TN = lambda L: f"{L} = {TASKS[L]!r}" if L else "None"   # label: letter + task name
# q-transfer predictions. set progress: first task of the list DST's goal heads read (KV's with kv, else DST's own)
# done in neither DST nor SRC · count: DST slot |SRC_DONE| on DST's own listing
_PLIST = KV_ORDER if USE_KV else DST_ORDER
H1_NEXT = next((L for L in _PLIST if L not in set(DST_DONE) | set(SRC_DONE)), None)
COUNT_NEXT = DST_ORDER[len(SRC_DONE)] if len(SRC_DONE) < len(DST_ORDER) else None
if USE_KV:
    assert set(DST_DONE) <= set(KV_ORDER) and all(TASKS[L] in by for L in KV_ORDER), "KV_ORDER must contain DST_DONE"
    KV_NEXT = next(L for L in KV_ORDER if L not in DST_DONE)                         # kv-only expectation
    print(f"KV {{{','.join(KV_ORDER)}}} · kv-only expects {TN(KV_NEXT)} · kv+q: set progress -> {TN(H1_NEXT)}, "
          f"count -> {TN(COUNT_NEXT)} (SRC identity -> {TN(TARGET)})")
print(f"q-transfer predicts: set progress -> {TN(H1_NEXT)} · count -> {TN(COUNT_NEXT)}")
print(f"{len(GOAL_HEADS)} goal heads · SRC target {TN(TARGET)} · DST frontier {TN(FRONTIER)} · "
      f"target in DST: {TARGET in DST_ORDER} · N_STEER={N_STEER}")
for L in sorted(set(SRC_ORDER + DST_ORDER + (KV_ORDER if USE_KV else ""))):
    print(f"  {L} = {TASKS[L]:<22} {by[TASKS[L]].description}")

q-transfer predicts: set progress -> D = 'count_starts_vowel' · count -> D = 'count_starts_vowel'
250 goal heads · SRC target C = 'distinct_count' · DST frontier B = 'count_long' · target in DST: False · N_STEER=32
  A = shortest_word_length   the length of the shortest word
  B = count_long             the number of words longer than five characters
  C = distinct_count         the number of distinct words
  D = count_starts_vowel     the number of words that start with a vowel


## SRC and DST prompts

Prompts as the model sees them, with the pivot marked. With `USE_KV=True`, p_dst and p_kv are shown **aligned**:
task-list span = yellow, the exact tokens whose k+v move = orange, spliced `pad` newlines = `⏎pad` (grey),
pivot = salmon.

In [37]:
# ── SRC / DST prompts ────────────────────────────────────────────────────────
class SeqQTI(DATA.QTransferInstance):
    """`letters` naming is SEQUENTIAL (DATA._seq_letter): prefilled lines are a, b, c, ... in the order they are
    written and the frontier stub takes the next letter — never the task's list position. (The base
    QTransferInstance is still positional, which leaks list position whenever the prefill is not a list prefix.)"""
    def var_of(self, pred_id):
        if self.naming != "letters":
            return super().var_of(pred_id)
        return DATA._seq_letter(self.task.predicates, self.src_progress, pred_id)


def make_inst(order, done, target):
    task = DATA.dataclasses.replace(bank, id="xtask", predicates=tuple(by[TASKS[L]] for L in order))
    ids = [TASKS[L] for L in order if L in done]      # written in list order (matches _seq_letter)
    return SeqQTI(task=task, var_name=bank.var_names[0], presentation="listed",
                                  src_lang="python", dst_lang="python", src_progress=ids, dst_progress=ids,
                                  expected_next=TASKS[target], naming=NAMING, meta={"datatype": DATATYPE})


src_inst = make_inst(SRC_ORDER, SRC_DONE, TARGET)
dst_inst = make_inst(DST_ORDER, DST_DONE, FRONTIER)
def build_prompt(inst, parts, next_task):
    q, pf = parts
    if PIVOT == "var_eq":            # stub the next task's (sequential) variable up to " =" -> pivot = " ="
        pf = pf + f"{inst.var_of(TASKS[next_task])} ="
    elif PIVOT != "indent":
        raise ValueError(f"PIVOT must be 'indent' or 'var_eq', got {PIVOT!r}")
    return get_prompt(q, DATA._finalize_prefill(pf))


src_prompt = build_prompt(src_inst, src_inst.src_parts(), TARGET)
dst_prompt = build_prompt(dst_inst, dst_inst.dst_parts(), FRONTIER)
piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)

tasks_str = lambda order: "{" + ",".join(order) + "}"
print(f"pivot token: SRC {mt.tokenizer.decode(src_tok['input_ids'][0][piv_src:piv_src+1])!r} · "
      f"DST {mt.tokenizer.decode(dst_tok['input_ids'][0][piv_dst:piv_dst+1])!r}")
show(src_prompt, f"SRC · {tasks_str(SRC_ORDER)} · {','.join(SRC_DONE)} done · target {TN(TARGET)} · pivot@{piv_src}")
if not USE_KV:                                       # with kv, DST is shown aligned (with spans) below
    show(dst_prompt, f"DST · {tasks_str(DST_ORDER)} · {','.join(DST_DONE)} done · frontier {TN(FRONTIER)} · pivot@{piv_dst}")

# ── optional KV prompt (04.3): its task-list k+v -> DST's task list ───────────
dst_tok_clean, piv_dst_clean = dst_tok, piv_dst      # clean DST, never padded
task_text = lambda order: "\n".join(f"({i + 1}) {by[TASKS[L]].description}" for i, L in enumerate(order))


def item_positions(prompt, order):
    return {L: list(range(*token_span(prompt, by[TASKS[L]].description))) for L in order}


PATCHED_ITEMS = item_positions(dst_prompt, DST_ORDER)   # {letter: DST positions} the patched runs' gaze is read over
_FROM_DST = {}                                           # DST position -> p_kv position, inside the kv window
PATCHED_LABEL = "DST tasks"


def show_spans(tok, title, task_span=None, kv_span=None, pad=(), piv=-1):
    """Render a tokenized prompt exactly as the model sees it (from 04.3): task_span -> bright_yellow,
    kv_span -> black on dark_orange (the exact tokens whose k+v move), pad -> the spliced "\\n" alignment tokens
    (black on grey62, shown as ⏎pad), pivot -> salmon. Priority pivot > pad > kv > task."""
    ids = tok["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    pad = set(pad)
    # sentinel markers, escape the joined string once, then swap in rich tags (per-token escaping drops brackets)
    PO, PC, DO, DC, KO, KC, TO, TC = "\x00<", "\x00>", "\x03<", "\x03>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in pad:
            s = DO + "⏎pad" + DC + "\n"
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    for a, b, tag in ((PO, PC, "bold black on salmon1"), (DO, DC, "black on grey62"),
                      (KO, KC, "bold black on dark_orange"), (TO, TC, "bright_yellow")):
        txt = txt.replace(a, f"[{tag}]").replace(b, f"[/{tag}]")
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))


def patched_tok_text(j):
    # token at DST position j as the patched runs' goal heads see it: inside the kv window that is p_kv's token
    # (its k+v were transplanted there), elsewhere DST's own
    if j in _FROM_DST:
        return mt.tokenizer.decode([int(kv_tok["input_ids"][0][_FROM_DST[j]])])
    return mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])])


if USE_KV:
    kv_inst = make_inst(KV_ORDER, DST_DONE, KV_NEXT)
    kv_prompt = build_prompt(kv_inst, kv_inst.dst_parts(), KV_NEXT)
    kv_task, dst_task = task_text(KV_ORDER), task_text(DST_ORDER)
    KV0, DST0 = token_span(kv_prompt, kv_task), token_span(dst_prompt, dst_task)
    kv_tok, dst_tok, KV_POS, DST_POS, KV_SPAN, DST_SPAN = align_kv_spans(
        mt, kv_prompt, dst_prompt, KV0, DST0, mode=KV_ALIGN, src_text=kv_task, dst_text=dst_task)
    piv_dst = dst_tok["input_ids"].shape[1] - 1          # patched runs use the (possibly padded) aligned DST
    _kvh = heads_per_layer(KV_HEADS, "k", layers=set(FUNC._layers_of(KV_HEADS)))
    KV_K = cache_src_proj(kv_tok, KV_POS, sorted(_kvh), "k")
    KV_V = cache_src_proj(kv_tok, KV_POS, sorted(_kvh), "v")
    # KV items -> DST positions through the aligned window (pad splices newlines right before the shorter span)
    _kv_pad = (KV_SPAN[1] - KV_SPAN[0]) - (KV0[1] - KV0[0])
    _to_dst = dict(zip(KV_POS, DST_POS))
    _FROM_DST = dict(zip(DST_POS, KV_POS))
    PATCHED_ITEMS = {L: [_to_dst[p + _kv_pad] for p in ps if p + _kv_pad in _to_dst]
                     for L, ps in item_positions(kv_prompt, KV_ORDER).items()}
    PATCHED_LABEL = "p_kv tasks"
    # alignment pads: "pad" splices newline ids right BEFORE the shorter span -> the first n positions of that span
    _dst_pad = (DST_SPAN[1] - DST_SPAN[0]) - (DST0[1] - DST0[0])
    KV_PADS = list(range(KV_SPAN[0], KV_SPAN[0] + _kv_pad)) if KV_ALIGN == "pad" else []
    DST_PADS = list(range(DST_SPAN[0], DST_SPAN[0] + _dst_pad)) if KV_ALIGN == "pad" else []
    print(f"align={KV_ALIGN}: {len(KV_POS)} transplanted tokens · pads: p_kv {len(KV_PADS)} · p_dst {len(DST_PADS)}"
          f" · same positions: {KV_POS == DST_POS}")
    show_spans(dst_tok, f"p_dst · {tasks_str(DST_ORDER)} · {','.join(DST_DONE)} done · frontier {TN(FRONTIER)} · "
                        f"k/v PATCHED-TO (aligned) · pivot@{piv_dst}",
               task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), pad=DST_PADS)
    show_spans(kv_tok, f"p_kv · {tasks_str(KV_ORDER)} · kv-only expects {TN(KV_NEXT)} · k/v CACHED-FROM (aligned)",
               task_span=KV_SPAN, kv_span=(KV_POS[0], KV_POS[-1] + 1), pad=KV_PADS)

pivot token: SRC ' =' · DST ' ='


╭─ SRC · {A,C,B} · A,B done · target C = 'distinct_count' · pivot@141 ────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the length of the shortest word\n                                                                           │
│ (2) the number of distinct words\n                                                                              │
│ (3) the number of words longer than five characters\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = min(len(w) for w in words)\n                                                                            │
│     b = sum(1 for w in words if len(w) > 5)\n                                                                   │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · {A,B,D} · A done · frontier B = 'count_long' · pivot@127 ────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the length of the shortest word\n                                                                           │
│ (2) the number of words longer than five characters\n                                                           │
│ (3) the number of words that start with a vowel\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = min(len(w) for w in words)\n                                                                            │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Runs — source / destination / (kv-only) / patched

| run | what runs | expect |
|---|---|---|
| source | clean p_src | C (SRC's target) |
| destination | clean, unpadded p_dst | DST's frontier |
| kv-only (`USE_KV`) | p_dst + p_kv's task-list k+v (+ boost) | p_kv's next (B) |
| patched | (kv +) SRC query at `PREFILL_ANCHORS`, re-clamped (`DECODE_Q`) for `N_STEER − 1` decode steps | `EXPECT_PATCHED` |

Each run: greedy generation + the GoalScope read-out ("what are you doing?") at its pivot. The patched read-out uses
the **prefill** patch only (`IFACE` = kv + the anchor q-patch + boost) — the decode re-clamps do not reach the pivot.

In [38]:
# ── source / destination / patched runs: generation + GoalScope read-out at the pivot ──
Q_LAYERS = sorted({l for l, _ in PATCH_HEADS})
_qh = heads_per_layer(PATCH_HEADS, "q")

# anchors = the last tokens of the prefill: "    c =" -> ['   ', ' c', ' ='] (indent, var, eq); indent pivot -> ['    ']
ANCHOR_OFFSET = {"indent": -3, "var": -2, "eq": -1} if PIVOT == "var_eq" else {"indent": -1}
assert set(PREFILL_ANCHORS) <= set(ANCHOR_OFFSET), f"PREFILL_ANCHORS {PREFILL_ANCHORS} need PIVOT='var_eq'"
anchor_pos = lambda tok: {a: tok["input_ids"].shape[1] + o for a, o in ANCHOR_OFFSET.items()}
SRC_ANCH, DST_ANCH = anchor_pos(src_tok), anchor_pos(dst_tok)
_tok_at = lambda tok, p: mt.tokenizer.decode([int(tok["input_ids"][0][p])])
for side, tok, anch in (("SRC", src_tok, SRC_ANCH), ("DST", dst_tok, DST_ANCH)):
    print(f"{side} anchors: " + "  ".join(f"{a}@{p}={_tok_at(tok, p)!r}" for a, p in anch.items()))
if PIVOT == "var_eq":
    assert all(_tok_at(t, a["indent"]).strip() == "" and _tok_at(t, a["eq"]).strip() == "="
               for t, a in ((src_tok, SRC_ANCH), (dst_tok, DST_ANCH))), "anchor offsets do not match the tokens"

# SRC queries (pre-RoPE) at every anchor: {layer: (n_anchors, width)}, rows in ANCHOR_OFFSET order
_anchors = list(ANCHOR_OFFSET)
q_src_all = cache_src_proj(src_prompt, [SRC_ANCH[a] for a in _anchors], Q_LAYERS, "q")
q_rows = lambda names: {l: c[[_anchors.index(a) for a in names]] for l, c in q_src_all.items()}

# prefill: row r of the cache goes to DST position r — indent->indent, var->var, eq->eq
PF = [dict(which="q", cache=q_rows(PREFILL_ANCHORS), heads=_qh,
           positions=[DST_ANCH[a] for a in PREFILL_ANCHORS])]
# decode: one query re-clamped at the current token for the first N_STEER-1 decode steps
if DECODE_Q == "eq" or PIVOT == "indent":
    q_dec = q_rows([_anchors[-1]])                                       # the " =" (or indent) query
elif DECODE_Q == "mean":
    q_dec = {l: c.mean(0, keepdim=True) for l, c in q_src_all.items()}  # mean of indent / var / eq
else:
    raise ValueError(f"DECODE_Q must be 'eq' or 'mean', got {DECODE_Q!r}")
DC = [dict(which="q", cache=q_dec, heads=_qh, positions=[-1])]
from contextlib import ExitStack


def patched_ctx(pf=None, boost=True):
    # stack: kv transplant (USE_KV) + query patch `pf` + the 04.3 OV boost (USE_KV only) — for ONE patched forward
    s = ExitStack()
    if USE_KV:
        s.enter_context(headwise_kv_patch(mt, KV_HEADS, KV_K, KV_V, DST_POS))
    if pf:
        s.enter_context(proj_patch_hooks(mt, pf))
    if boost and USE_KV and KV_ALPHA != 1.0:
        s.enter_context(boost_head_ov(mt, sorted(set(KV_HEADS) | set(PATCH_HEADS)), KV_ALPHA, positions=(-1,)))
    return s


def ro_iface(pf=None, boost=True):
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with patched_ctx(pf, boost=boost):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if


IFACE = ro_iface(PF)                       # kv (if USE_KV) + prefill q-patch (+ boost), for read-outs
EXPECT_PATCHED = (f"set+count {TN(H1_NEXT)}" if H1_NEXT == COUNT_NEXT
                  else f"set {TN(H1_NEXT)} / count {TN(COUNT_NEXT)}")      # what the patched run is judged against
print(f"prefill patch at {list(PREFILL_ANCHORS)} · decode query = {DECODE_Q if PIVOT == 'var_eq' else 'indent'} "
      f"· N_STEER={N_STEER}")

GEN, RO = {}, {}
GEN["source"] = greedy(src_tok, max_new=MAX_NEW)
rich_print(f"[red1]source · clean SRC (target {TN(TARGET)}):[/red1]\n{escape(GEN['source'])}")
RO["source"] = goalscope_readout("   ↳ goalscope · source pivot", src_prompt, piv_src, heads=READOUT_HEADS, **RO_KW)

GEN["destination"] = greedy(dst_tok_clean, max_new=MAX_NEW)
rich_print(f"[sea_green3]destination · clean DST (frontier {TN(FRONTIER)}):[/sea_green3]\n{escape(GEN['destination'])}")
RO["destination"] = goalscope_readout("   ↳ goalscope · destination pivot", dst_prompt, piv_dst_clean,
                                      heads=READOUT_HEADS, **RO_KW)

if USE_KV:
    with patched_ctx():
        GEN["kv"] = greedy(dst_tok, max_new=MAX_NEW)
    rich_print(f"[orange3]kv-only · DST + KV task-list k+v (expect {TN(KV_NEXT)}):[/orange3]\n{escape(GEN['kv'])}")
    RO["kv"] = goalscope_readout("   ↳ goalscope · kv-only pivot", dst_tok, piv_dst, heads=READOUT_HEADS,
                                 iface=ro_iface(), **RO_KW)

with patched_ctx():                        # kv + boost; the query patch itself comes from greedy's specs
    GEN["patched"] = greedy(dst_tok, prefill_specs=PF, decode_specs=DC, n_steer=N_STEER, max_new=MAX_NEW)
rich_print(f"[salmon1]patched · DST{' + KV k+v' if USE_KV else ''} + SRC query (expect {EXPECT_PATCHED}; "
           f"N_STEER={N_STEER}):[/salmon1]\n"
           f"{escape(GEN['patched'])}")
RO["patched"] = goalscope_readout(f"   ↳ goalscope · patched pivot (expect {EXPECT_PATCHED})", dst_tok, piv_dst,
                                  heads=READOUT_HEADS,
                                  iface=IFACE, **RO_KW)

SRC anchors: indent@139='   '  var@140=' c'  eq@141=' ='
DST anchors: indent@125='   '  var@126=' b'  eq@127=' ='
prefill patch at ['indent', 'var', 'eq'] · decode query = eq · N_STEER=32


source · clean SRC (target C = 'distinct_count'):
 len(set(words))
    return a, c, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source pivot  pos=141 ' =' => I'm counting unique elements in a list using a set.

destination · clean DST (frontier B = 'count_long'):
 sum(1 for w in words if len(w) > 5)
    c = sum(1 for w in words if w[0].lower() in 'aeiou')
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination pivot  pos=127 ' =' => I'm counting lines where the length of each word is greater 
than 5.

patched · DST + SRC query (expect set+count D = 'count_starts_vowel'; N_STEER=32):
 sum(1 for w in words if w[0].isalpha() and w[0].lower() in 'aeiou')
    c = sum(1 for w in words if len(w) > 5)
    return a, c, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched pivot (expect set+count D = 'count_starts_vowel')  pos=127 ' =' => I'm counting how many 
lines start with a vowel.

## Causality read-out — steer the pivot's goal encoding into an empty function

Same GoalScope steering, but instead of asking "what are you doing?", the goal-head encoding at the pivot is steered
into an information-less function stub and the model continues the code (mirrors `eval_goalscope --mode causality`,
whose stub is the C++ `bool check(auto value){`). The stub reuses DST's variable name and datatype and **always ends
at `return`**, whatever `PIVOT` is:
```
def func({var}: {type}):
    return
```
Read-outs for source, destination, kv-only (`USE_KV`) and patched (`IFACE`: kv + prefill q-patch + boost). The
continuation is the transferred *idea* — does the patched one implement the predicted task?

In [39]:
from src.goals.goalscope import apply_goalscope, GOALSCOPE_SYS

CAUSAL_MAX_NEW = 64
CAUSAL_INSTRUCTION_PY = ("You are implementing a simple Python function. Just write the function body, without any "
                         "comments or explanations.")
# Python stub from DST's own signature (same variable name + datatype as the destination prompt)
CAUSAL_PREFILL = DATA._signature(DATATYPE, dst_inst.var_name, "python") + "\n    return"


def _chat(user, assistant):
    msgs = [{"role": "system", "content": GOALSCOPE_SYS}, {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    kw = dict(add_generation_prompt=False, continue_final_message=True, tokenize=False)
    try:
        return mt.tokenizer.apply_chat_template(msgs, chat_template_kwargs={"enable_thinking": False}, **kw)
    except TypeError:
        return mt.tokenizer.apply_chat_template(msgs, **kw)


CAUSAL_GS_PROMPT = _chat(CAUSAL_INSTRUCTION_PY, CAUSAL_PREFILL)


def causal_readout(tag, prompt, pivot, iface=None):
    res = apply_goalscope(prompt=prompt, mt=mt, selected_heads=list(READOUT_HEADS), strategy=RO_KW["strategy"],
                          alpha=RO_KW["alpha"], steer_layer_range=RO_KW["layers"], batch_size=1,
                          max_new_tokens=CAUSAL_MAX_NEW, goalscope_prompt=CAUSAL_GS_PROMPT,
                          token_positions=[pivot], include_noncode=False, show=False, patch_interface=iface)
    out = res[0]["patched_gen"]
    rich_print(f"[bold]{tag}[/bold]  pos={res[0]['pos']} {res[0]['token']!r} =>\n"
               f"[grey58]{escape(CAUSAL_PREFILL)}[/grey58][italic]{escape(out)}[/italic]")
    return out


rich_print(f"[grey58]causal stub:[/grey58] {escape(CAUSAL_PREFILL)!s}")
CAUSAL = {
    "source":      causal_readout(f"source (target {TN(TARGET)})", src_prompt, piv_src),
    "destination": causal_readout(f"destination (frontier {TN(FRONTIER)})", dst_prompt, piv_dst_clean),
    "patched":     causal_readout(f"patched{' (kv + q)' if USE_KV else ''} (expect {EXPECT_PATCHED})", dst_tok, piv_dst,
                                  iface=IFACE),
}
if USE_KV:
    CAUSAL["kv"] = causal_readout(f"kv-only (expect {TN(KV_NEXT)})", dst_tok, piv_dst, iface=ro_iface())

causal stub: def func(words: list[str]):
    return

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

source (target C = 'distinct_count')  pos=141 ' =' =>
def func(words: list[str]):
    return len(set(words))

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

destination (frontier B = 'count_long')  pos=127 ' =' =>
def func(words: list[str]):
    return sum(1 for word in words if len(word) > 5)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

patched (expect set+count D = 'count_starts_vowel')  pos=127 ' =' =>
def func(words: list[str]):
    return sum(1 for word in words if word[0].lower() in 'aeiou')

## Gaze — where does the pivot look in the task list?

Mean **value-weighted** attention from the pivot onto each listed task's description, averaged over `GAZE_HEADS`.
Measured under the plain patches — **no OV boost** (it scales the write, not the attention), as in 04.3.

| run | read over | patches |
|---|---|---|
| source | p_src's list | none |
| destination | p_dst's own list (clean, unpadded) | none |
| kv-only (`USE_KV`) | DST's task-list positions, as **p_kv's items** | kv |
| patched | DST's task-list positions — p_kv's items with kv, else DST's | (kv +) the **prefill** q-patch at `PREFILL_ANCHORS` |

With kv on, DST's task-list positions hold p_kv's key+value on the goal heads, so the per-task columns are p_kv's
tasks (mapped through the `pad` alignment) and the token-level plots show **p_kv's text**. The patched gaze shows
what the transplanted pivot query attends to; the decode re-clamps are not involved.

Plots: the table (argmax in bold), then token-level distributions — clean destination; kv-only; patched; and
patched − clean (without kv only, when both are over DST's own list).

In [40]:
def pivot_gaze(tok, items, piv, iface=None):
    # mean value-weighted pivot attention (GAZE_HEADS) per item; items = {letter: positions in `tok`}
    kw = {} if iface is None else dict(patch_interface=iface)
    info = get_attention_matrices(tok, mt, value_weighted=True, **kw)
    info = info[0] if isinstance(info, list) else info
    span = list(range(min(p[0] for p in items.values()), max(p[-1] for p in items.values()) + 1))
    a = pivot_attn(info, list(GAZE_HEADS), span, piv)                  # (len(span),)
    at = {p: v for p, v in zip(span, a)}
    per_item = {L: float(np.mean([at[p] for p in ps])) for L, ps in items.items()}
    return per_item, span, a

# gaze is measured under the plain patch (no OV boost — it scales the write, not the attention), as in 04.3
GAZE = {
    "source":      pivot_gaze(src_tok, item_positions(src_prompt, SRC_ORDER), piv_src),
    "destination": pivot_gaze(dst_tok_clean, item_positions(dst_prompt, DST_ORDER), piv_dst_clean),
}
if USE_KV:
    GAZE["kv"] = pivot_gaze(dst_tok, PATCHED_ITEMS, piv_dst, iface=ro_iface(boost=False))
GAZE["patched"] = pivot_gaze(dst_tok, PATCHED_ITEMS, piv_dst, iface=ro_iface(PF, boost=False))

t = Table(title=f"pivot gaze per task (mean value-weighted attn, {len(GAZE_HEADS)} heads) — argmax in bold"
                + (" · kv / patched read over p_kv's task span" if USE_KV else ""), highlight=False)
t.add_column("run"); t.add_column("expect")
for k in range(max(len(SRC_ORDER), len(DST_ORDER), len(KV_ORDER) if USE_KV else 0)):
    t.add_column(f"task {k + 1}", justify="right")
rows = [("source", TN(TARGET)), ("destination", TN(FRONTIER))]
rows += ([("kv", TN(KV_NEXT))] if USE_KV else []) + [("patched", EXPECT_PATCHED)]
for run, expect in rows:
    g_ = GAZE[run][0]; top = max(g_, key=g_.get)
    t.add_row(run, expect, *[f"[bold]{L}={v:.4f}[/bold]" if L == top else f"{L}={v:.4f}" for L, v in g_.items()])
rich_print(t)

# token-level views: clean DST over its own list; patched over DST's (kv-patched) task span
_, span_c, a_clean = GAZE["destination"]
toks_c = [mt.tokenizer.decode([int(dst_tok_clean["input_ids"][0][j])]) for j in span_c]
show_delta_tokens(toks_c, a_clean, "destination · pivot → DST tasks (clean)")
_, span, a_patched = GAZE["patched"]
toks = [patched_tok_text(j) for j in span]            # p_kv's task text when USE_KV
if USE_KV:
    show_delta_tokens(toks, GAZE["kv"][2], "kv-only · pivot → p_kv tasks")
show_delta_tokens(toks, a_patched, f"patched · pivot → {PATCHED_LABEL}")
if not USE_KV:
    show_delta_tokens(toks, a_patched - a_clean, "patched − clean · pivot → DST tasks")

     pivot gaze per task (mean value-weighted attn, 250 heads) — argmax in bold      
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ run         ┃ expect                             ┃   task 1 ┃   task 2 ┃   task 3 ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ source      │ C = 'distinct_count'               │ A=0.0314 │ C=0.1878 │ B=0.0580 │
│ destination │ B = 'count_long'                   │ A=0.0367 │ B=0.1539 │ D=0.0691 │
│ patched     │ set+count D = 'count_starts_vowel' │ A=0.0442 │ B=0.0777 │ D=0.1281 │
└─────────────┴────────────────────────────────────┴──────────┴──────────┴──────────┘

destination · pivot → DST tasks (clean)   (green +Δ / red −Δ, |Δ|max=0.460)

patched · pivot → DST tasks   (green +Δ / red −Δ, |Δ|max=0.421)

patched − clean · pivot → DST tasks   (green +Δ / red −Δ, |Δ|max=0.312)

## Closed-loop query transfer — regenerate the goal heads' query at every step

The re-clamp run above writes the **same** SRC query into the current token at every decode step. Under the NRGS view
the goal persists but the query changes token by token ("given this goal and what I have written, what do I look
for next?"). The closed loop keeps the goal fixed and **recomputes** the query at every step:

1. **Prefill (i)** — the patched DST forward with the SRC queries at `PREFILL_ANCHORS` (plus kv + boost when
   `USE_KV`) → first token `t1`.
2. **Goal encoding (ii)** — the goal heads' OV write (`READOUT_HEADS`) **in that same patched prefill**, at ` =`
   (`GOAL_ENC="eq"`) or averaged over `PREFILL_ANCHORS` (`"mean"`). Never from the clean SRC. Fixed for the whole loop.
3. **Probe** — an information-less prompt with DST's signature, ending at the bare trailing indent
   (`PIVOT="indent"`) or `    return` (`PIVOT="var_eq"`). The goal encoding is steered in with `additive_norm`
   (α `PROBE_ALPHA`, residual after layers `PROBE_LAYERS` = 35–45) at the probe pivot and, with `PROBE_STEER="all"`,
   at every appended token (`"pivot"`: the pivot only). The probe is never kv- or q-patched.
4. **Each step k ≥ 1 (iii)** — append `t_k` to the probe; read its query at `t_k` on the goal heads (`q_caus`);
   write `q_caus` into the patched run's goal-head query at `t_k` (every other head keeps its own; kv + boost stay
   on); generate `t_{k+1}`. Stop at the first newline or `LOOP_MAX` tokens.

So the goal heads' query comes from SRC at the prefill anchors, and from the steered probe at every generated token.
Output: a per-step table (`t_k` and cos(`q_caus`, the re-clamp run's fixed decode query), mean over heads) and the
first line of the closed loop next to the clean destination and the re-clamp run.

In [41]:
# ── knobs ────────────────────────────────────────────────────────────────────
GOAL_ENC    = "mean"         # "eq": goal heads' OV write at DST " =" · "mean": mean over PREFILL_ANCHORS
PROBE_STEER = "all"        # "pivot": steer only the probe's " return" · "all": also every appended token
LOOP_MAX    = 25           # max generated tokens (also stops at the first newline)
PROBE_ALPHA, PROBE_LAYERS = 0.1, (35, 46)   # additive_norm into the residual after layers range(35, 46)

import baukit
from contextlib import contextmanager, nullcontext
from src.goals.goalscope import GOALSCOPE_SYS
from src.goals.head_loc_attn import collect_head_steering_vecs

# (2) goal encoding — goal heads' OV write in the q-patched DST prefill (IFACE = the PREFILL_ANCHORS patch)
_anch = list(PREFILL_ANCHORS)
_ov = collect_head_steering_vecs(mt=mt, src_tokenized=dst_tok, selected_heads=list(READOUT_HEADS),
                                 token_positions=[DST_ANCH[a] for a in _anch], patch_interface=IFACE)[0]
if GOAL_ENC == "eq":
    assert "eq" in _anch, "GOAL_ENC='eq' needs 'eq' in PREFILL_ANCHORS"
    _g = _ov[_anch.index("eq")]
elif GOAL_ENC == "mean":
    _g = _ov.mean(0)
else:
    raise ValueError(f"GOAL_ENC must be 'eq' or 'mean', got {GOAL_ENC!r}")
G_UNIT = (_g / _g.norm()).float()                     # additive_norm: unit direction, magnitude from ||h||

# probe: information-less function stub, same variable name + datatype as DST
# pivot shape follows PIVOT: "indent" -> bare trailing indent (else the model writes `return b = ...`),
# "var_eq" -> `return`
PROBE_PREFILL = DATA._signature(DATATYPE, dst_inst.var_name, "python") + ("\n    return" if PIVOT == "var_eq" else "\n    ")
PROBE_INSTRUCTION = ("You are implementing a simple Python function. Just write the function body, without any "
                     "comments or explanations.")


def _probe_chat(user, assistant):
    msgs = [{"role": "system", "content": GOALSCOPE_SYS}, {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    kw = dict(add_generation_prompt=False, continue_final_message=True, tokenize=False)
    try:
        return mt.tokenizer.apply_chat_template(msgs, chat_template_kwargs={"enable_thinking": False}, **kw)
    except TypeError:
        return mt.tokenizer.apply_chat_template(msgs, **kw)


probe_prompt = _probe_chat(PROBE_INSTRUCTION, PROBE_PREFILL)
probe_tok = prepare_input(prompts=probe_prompt, tokenizer=mt)
piv_probe = probe_tok["input_ids"].shape[1] - 1
print(f"probe pivot {mt.tokenizer.decode(probe_tok['input_ids'][0][piv_probe:piv_probe + 1])!r} · "
      f"goal enc = {GOAL_ENC} over {_anch} · probe steer = {PROBE_STEER} · layers {PROBE_LAYERS}")

_STEER_NAMES = [mt.layer_name_format.format(l) for l in range(*PROBE_LAYERS)]
_Q_NAMES = {l: FUNC._proj_name(l, "q", mt) for l in Q_LAYERS}


@contextmanager
def goal_steer(positions):
    # h <- h + alpha * ||h|| * g_unit at `positions` of the current forward, residual after each steer layer
    def edit(output, layer):
        h = output[0] if isinstance(output, tuple) else output
        for p in positions:
            v = h[:, p, :]
            h[:, p, :] = v + PROBE_ALPHA * v.float().norm(dim=-1, keepdim=True).to(v.dtype) \
                * G_UNIT.to(v.device, v.dtype)
        return output
    with baukit.TraceDict(mt._model, _STEER_NAMES, edit_output=edit):
        yield


def _ext(am):
    return torch.cat([am, torch.ones_like(am[:, :1])], 1)


def _head_cos(qa, qb):
    # mean over PATCH_HEADS of cos(q_a, q_b) per head slice
    hd, out = mt.head_dim, []
    for l, h in PATCH_HEADS:
        a, b = qa[l][0, h * hd:(h + 1) * hd], qb[l][0, h * hd:(h + 1) * hd]
        out.append(float(torch.nn.functional.cosine_similarity(a, b, dim=0)))
    return float(np.mean(out))


@torch.no_grad()
def closed_loop(verbose=True):
    eos = mt.tokenizer.eos_token_id
    # (1) patched-run prefill: SRC queries at the anchors -> t1
    ids, am = dst_tok["input_ids"], dst_tok["attention_mask"]
    with patched_ctx(pf=PF):
        out = mt._model(input_ids=ids, attention_mask=am, use_cache=True)
    pkv_d, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
    # probe prefill: goal steered at " return"
    p_am = probe_tok["attention_mask"]
    with goal_steer([piv_probe]):
        pout = mt._model(input_ids=probe_tok["input_ids"], attention_mask=p_am, use_cache=True)
    pkv_p = pout.past_key_values

    gen, steps = [int(nxt)], []
    while len(gen) < LOOP_MAX and int(nxt) != eos and "\n" not in mt.tokenizer.decode(gen):
        # (3) probe consumes t_k (goal-steered if PROBE_STEER="all"); read q_caus at t_k on the goal heads
        p_am = _ext(p_am)
        ctx = goal_steer([-1]) if PROBE_STEER == "all" else nullcontext()
        with ctx, baukit.TraceDict(mt._model, list(_Q_NAMES.values()), retain_output=True) as tr:
            pout = mt._model(input_ids=nxt, attention_mask=p_am, past_key_values=pkv_p, use_cache=True)
        pkv_p = pout.past_key_values
        q_caus = {l: tr[n].output[0, -1:].detach().float().cpu() for l, n in _Q_NAMES.items()}
        # patched run: q_caus replaces the goal heads' query at t_k -> t_{k+1}
        am = _ext(am)
        with patched_ctx(pf=[dict(which="q", cache=q_caus, heads=_qh, positions=[-1])]):
            out = mt._model(input_ids=nxt, attention_mask=am, past_key_values=pkv_d, use_cache=True)
        pkv_d = out.past_key_values
        steps.append(dict(k=len(gen), tok=mt.tokenizer.decode(gen[-1:]), cos_q_dec=_head_cos(q_caus, q_dec), q=q_caus))
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
        gen.append(int(nxt))
    text = mt.tokenizer.decode(gen, skip_special_tokens=True)
    if verbose:
        t = Table(title=f"closed loop · per step: token t_k whose query is replaced · cos(q_caus, decode q of the "
                        f"re-clamp run), mean over {len(PATCH_HEADS)} heads", highlight=False)
        t.add_column("k", justify="right"); t.add_column("t_k"); t.add_column("cos", justify="right")
        for s in steps:
            t.add_row(str(s["k"]), repr(s["tok"]), f"{s['cos_q_dec']:.3f}")
        rich_print(t)
    return text, steps, gen


GEN["closed_loop"], LOOP_STEPS, LOOP_IDS = closed_loop()
_line = lambda s: s.split("\n")[0]
rich_print(f"[sea_green3]destination (frontier {TN(FRONTIER)}):[/sea_green3] {escape(_line(GEN['destination']))}")
rich_print(f"[salmon1]q_src re-clamp (N_STEER={N_STEER}):[/salmon1]   {escape(_line(GEN['patched']))}")
rich_print(f"[bold magenta]closed loop (expect {EXPECT_PATCHED}):[/bold magenta]  {escape(_line(GEN['closed_loop']))}")

probe pivot ' return' · goal enc = mean over ['indent', 'var', 'eq'] · probe steer = all · layers (35, 46)


 closed loop · per step: 
token t_k whose query is 
 replaced · cos(q_caus,  
decode q of the re-clamp 
run), mean over 250 heads
┏━━━━┳━━━━━━━━━━┳━━━━━━━┓
┃  k ┃ t_k      ┃   cos ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━┩
│  1 │ ' sum'   │ 0.684 │
│  2 │ '('      │ 0.664 │
│  3 │ '1'      │ 0.672 │
│  4 │ ' for'   │ 0.654 │
│  5 │ ' w'     │ 0.623 │
│  6 │ ' in'    │ 0.690 │
│  7 │ ' words' │ 0.636 │
│  8 │ ' if'    │ 0.670 │
│  9 │ ' w'     │ 0.633 │
│ 10 │ '['      │ 0.607 │
│ 11 │ '0'      │ 0.614 │
│ 12 │ '].'     │ 0.632 │
│ 13 │ 'lower'  │ 0.613 │
│ 14 │ '()'     │ 0.620 │
│ 15 │ ' in'    │ 0.663 │
│ 16 │ " '"     │ 0.631 │
│ 17 │ 'ae'     │ 0.606 │
│ 18 │ 'iou'    │ 0.622 │
└────┴──────────┴───────┘

destination (frontier B = 'count_long'):  sum(1 for w in words if len(w) > 5)

q_src re-clamp (N_STEER=32):    sum(1 for w in words if w[0].isalpha() and w[0].lower() in 'aeiou')

closed loop (expect set+count D = 'count_starts_vowel'):   sum(1 for w in words if w[0].lower() in 'aeiou')

### Closed loop — read-outs of the goal encoding (ii) + gaze along the loop

* **Read-outs of the goal encoding itself** (`_g`, what drives `q_caus`): GoalScope ("what are you doing?") and a
  causality read-out on the **loop's probe prompt** (bare indent under `PIVOT="indent"`, `return` under
  `"var_eq"`), steered as in the loop (`PROBE_STEER`). Under `PIVOT="indent"` this prompt differs from the causality
  section's `return` stub, so the two continuations need not match even though the vector is the same.
* **Gaze of the patched run** at the pivot (prefill patch only) and at every loop token `t_k` (its goal-head query =
  `q_caus`), over the same positions as the gaze section (p_kv's items when `USE_KV`). Obtained by replaying
  DST + `t_1..t_m` with the identical patches (kv, prefill anchors, `q_caus` at each `t_k`; no boost), which
  reproduces the loop's forward. Plots: the per-step table, then the **actual** attention distribution at every
  `t_k`.

In [42]:
from src.goals.goalscope import head_additive_generate, _build_goalscope_prompt
from src.utils.typing import TokenizerOutput

# ── read-outs of the goal encoding ───────────────────────────────────────────
_ro = lambda prompt, n_steer: head_additive_generate(
    mt=mt, dest_inputs=prepare_input(prompts=prompt, tokenizer=mt), steering_vecs=_g.float(),
    layers=list(range(*PROBE_LAYERS)), max_new_tokens=64, strategy="additive_norm", n_steer_tokens=n_steer,
    alpha=PROBE_ALPHA)
LOOP_RO = {
    "goalscope": _ro(_build_goalscope_prompt(mt), None),
    "causality": _ro(probe_prompt, None if PROBE_STEER == "all" else 1),
}
rich_print(f"[bold]goal encoding ({GOAL_ENC}) · GoalScope[/bold] => [italic]{escape(LOOP_RO['goalscope'])}[/italic]")
rich_print(f"[bold]goal encoding ({GOAL_ENC}) · causality[/bold] =>\n[grey58]{escape(PROBE_PREFILL)}[/grey58]"
           f"[italic]{escape(LOOP_RO['causality'])}[/italic]")

# ── gaze along the loop ─────────────────────────────────────────────────────
L0 = dst_tok["input_ids"].shape[1]
m = len(LOOP_STEPS)        # steps whose query was replaced: t_1 .. t_m
replay = TokenizerOutput(data=dict(
    input_ids=torch.cat([dst_tok["input_ids"], torch.tensor([LOOP_IDS[:m]], device=dst_tok["input_ids"].device)], 1),
    attention_mask=torch.ones(1, L0 + m, dtype=dst_tok["attention_mask"].dtype,
                              device=dst_tok["attention_mask"].device)))
loop_spec = dict(which="q", heads=_qh, positions=[L0 + k for k in range(m)],
                 cache={l: torch.cat([s["q"][l] for s in LOOP_STEPS], 0) for l in Q_LAYERS})
_info = get_attention_matrices(replay, mt, value_weighted=True,
                               patch_interface=ro_iface(PF + [loop_spec], boost=False))
_info = _info[0] if isinstance(_info, list) else _info

_items = PATCHED_ITEMS                    # DST items (or KV items when USE_KV)
_span = list(range(min(p[0] for p in _items.values()), max(p[-1] for p in _items.values()) + 1))
def _per_item(a):
    at = dict(zip(_span, a)); return {L: float(np.mean([at[p] for p in ps])) for L, ps in _items.items()}

LOOP_GAZE = [("pivot " + repr(mt.tokenizer.decode(dst_tok["input_ids"][0][piv_dst:piv_dst + 1])),
              pivot_attn(_info, list(GAZE_HEADS), _span, piv_dst))]
LOOP_GAZE += [(f"t_{s['k']} {s['tok']!r}", pivot_attn(_info, list(GAZE_HEADS), _span, L0 + s["k"] - 1))
              for s in LOOP_STEPS]
t = Table(title=f"closed-loop gaze over {PATCHED_LABEL} (mean value-weighted attn, {len(GAZE_HEADS)} heads; frontier "
                f"{TN(FRONTIER)}, SRC target {TN(TARGET)}" + (f", kv-only {TN(KV_NEXT)}" if USE_KV else "") + f", expect {EXPECT_PATCHED}" + ") — argmax in bold", highlight=False)
t.add_column("query"); [t.add_column(L, justify="right") for L in _items]
for name, a in LOOP_GAZE:
    g_ = _per_item(a); top = max(g_, key=g_.get)
    t.add_row(name, *[f"[bold]{v:.4f}[/bold]" if L == top else f"{v:.4f}" for L, v in g_.items()])
rich_print(t)

_toks = [patched_tok_text(j) for j in _span]           # p_kv's task text when USE_KV
# actual gaze (not delta) of the goal heads at every loop step t_k, whose query was replaced by q_caus
for name, a in LOOP_GAZE[1:]:
    show_delta_tokens(_toks, a, f"closed loop · {name} (query = q_caus) → {PATCHED_LABEL}")

goal encoding (mean) · GoalScope => I'm counting lines where the condition `len(word) > 5` is true for words 
starting with a vowel and checking for words ending with 'e'.

goal encoding (mean) · causality =>
def func(words: list[str]):
    return sum(1 for word in words if len(word) > 3 and word[0] == 'a')
    count = 0
    for word in words:
        if len(word) > 3 and word[0] == 'a':
            count += 1
    return count

   closed-loop gaze over DST tasks (mean   
value-weighted attn, 250 heads; frontier B 
      = 'count_long', SRC target C =       
  'distinct_count', expect set+count D =   
  'count_starts_vowel') — argmax in bold   
┏━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ query        ┃      A ┃      B ┃      D ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ pivot ' ='   │ 0.0441 │ 0.0776 │ 0.1281 │
│ t_1 ' sum'   │ 0.0924 │ 0.1364 │ 0.1216 │
│ t_2 '('      │ 0.0839 │ 0.1598 │ 0.1437 │
│ t_3 '1'      │ 0.0603 │ 0.0872 │ 0.0841 │
│ t_4 ' for'   │ 0.1047 │ 0.1218 │ 0.1242 │
│ t_5 ' w'     │ 0.0588 │ 0.1332 │ 0.1353 │
│ t_6 ' in'    │ 0.0443 │ 0.0804 │ 0.0751 │
│ t_7 ' words' │ 0.0541 │ 0.1105 │ 0.0980 │
│ t_8 ' if'    │ 0.0701 │ 0.1503 │ 0.2427 │
│ t_9 ' w'     │ 0.0752 │ 0.1842 │ 0.2610 │
│ t_10 '['     │ 0.0523 │ 0.1392 │ 0.2309 │
│ t_11 '0'     │ 0.0518 │ 0.1094 │ 0.2244 │
│ t_12 '].'    │ 0.0429 │ 0.0956 │ 0.2758 │
│ t_13 'lower' │ 0.0316 │ 0.0689 │ 0.2030 │
│ t_14 '()'    │ 0.0504 │ 0.1080 │ 0.1537 │
│ t_15 ' in'   │ 0.0280 │ 0.0585 │ 0.2264 │
│ t_16 " '"    │ 0.0348 │ 0.0844 │ 0.2551 │
│ t_17 'ae'    │ 0.0302 │ 0.0699 │ 0.1126 │
│ t_18 'iou'   │ 0.0324 │ 0.0752 │ 0.0834 │
└──────────────┴────────┴────────┴────────┘

closed loop · t_1 ' sum' (query = q_caus) → DST tasks   (green +Δ / red −Δ, |Δ|max=0.541)

closed loop · t_2 '(' (query = q_caus) → DST tasks   (green +Δ / red −Δ, |Δ|max=0.705)

closed loop · t_3 '1' (query = q_caus) → DST tasks   (green +Δ / red −Δ, |Δ|max=0.278)

... (11 outputs left out)

closed loop · t_15 ' in' (query = q_caus) → DST tasks   (green +Δ / red −Δ, |Δ|max=1.612)

closed loop · t_16 " '" (query = q_caus) → DST tasks   (green +Δ / red −Δ, |Δ|max=1.798)

closed loop · t_17 'ae' (query = q_caus) → DST tasks   (green +Δ / red −Δ, |Δ|max=0.639)

closed loop · t_18 'iou' (query = q_caus) → DST tasks   (green +Δ / red −Δ, |Δ|max=0.379)